# TotalSpineSeg trên Kaggle GPU – 515 ảnh MRI sagittal

Phân đoạn + đánh số đốt sống / đĩa đệm (L1–L5, S, T12-L1 … L5-S, ống sống) cho 515 bệnh nhân.

**Trước khi chạy** (panel bên phải, *Settings*):
1. **Accelerator: GPU T4 x2** (không chọn P100 – PyTorch mới không còn hỗ trợ).
2. **Internet: On** (để cài thư viện và tải trọng số mô hình ~4 GB từ GitHub).
3. *Add Input* → dataset chứa `lumbar_sag_nifti.zip` (515 file `0001.nii.gz` …).

**Cách chạy:**
* Chạy **cell 1–4** (chế độ tương tác) để kiểm tra cài đặt và đo thời gian trên 3 ca.
* Nếu ổn: **Save Version → Save & Run All (Commit)** để chạy ngầm tối đa 12 giờ, không cần mở trình duyệt.
* Kết quả: `/kaggle/working/tss_sag_results.zip` (tab *Output* của version).
* Bị ngắt giữa chừng? Add Output của version cũ làm Input rồi chạy lại – ca đã xong sẽ được bỏ qua.


In [ ]:
# 1. Cấu hình
import os, glob, time, shutil, subprocess
from pathlib import Path

WORK = Path('/kaggle/working')
RES = WORK / 'tss_sag'                    # kết quả giữ lại (step2_output, step1_levels, preview)
TMP = Path('/tmp/tss')                    # thư mục tạm, KHÔNG lưu vào output
TSS_DATA = Path('/tmp/tss_data')          # trọng số mô hình (~4 GB) - không đưa vào output
BATCH = 25                                # số ca mỗi lần gọi totalspineseg
TIME_BUDGET_H = 11.0                      # dừng nhận lô mới sau 11 giờ (Kaggle cắt ở 12 giờ)
KEEP = ['step2_output', 'step1_levels', 'preview']

# In cấu trúc /kaggle/input (tối đa 3 cấp, 5 mục mỗi thư mục) để chẩn đoán
import re, zipfile
def show_tree(root='/kaggle/input', depth=3):
    for cur, dirs, files in os.walk(root, followlinks=True):
        lvl = cur[len(root):].count(os.sep)
        if lvl > depth:
            continue
        print('  ' * lvl + os.path.basename(cur) + '/', f'({len(files)} file)')
        for f in sorted(files)[:5]:
            print('  ' * (lvl + 1) + f)
show_tree()

# Tìm ảnh đầu vào. Kaggle có thể: (a) giải nén zip -> 0001.nii.gz, (b) giải nén cả .gz
# -> 0001.nii, hoặc (c) để nguyên file .zip. Xử lý cả 3 trường hợp.
PAT = re.compile(r'^\d{4}\.nii(\.gz)?$')
found = []
for cur, dirs, files in os.walk('/kaggle/input', followlinks=True):
    if 'step2_output' in cur or 'step1' in cur:
        continue
    found += [Path(cur) / f for f in files if PAT.match(f)]
if not found:
    zips = [Path(cur) / f for cur, _, fs in os.walk('/kaggle/input', followlinks=True)
            for f in fs if f.endswith('.zip')]
    assert zips, 'Không thấy ảnh .nii/.nii.gz hay file .zip nào - kiểm tra đã Add Input dataset chưa'
    print('Giải nén', zips[0])
    with zipfile.ZipFile(zips[0]) as z:
        z.extractall('/tmp/sag_in')
    found = [p for p in Path('/tmp/sag_in').rglob('*') if PAT.match(p.name)]
IN_DIR = found[0].parent
inputs = sorted(p for p in IN_DIR.iterdir() if PAT.match(p.name))
print('\nẢnh đầu vào:', IN_DIR, '-', len(inputs), 'file, vd', inputs[0].name)
assert len(inputs) == 515, f'Mong đợi 515 file, thấy {len(inputs)}'


In [ ]:
# 2. Cài đặt (~2-4 phút). kornia<0.8: bản 0.8.x bỏ kornia.core.Tensor mà auglab cần.
!pip install -q "totalspineseg[nnunetv2]" "kornia<0.8"
import torch
print('torch', torch.__version__, '| CUDA:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'KHÔNG CÓ GPU')
assert torch.cuda.is_available(), 'Chưa bật GPU: Settings -> Accelerator -> GPU T4 x2'


In [ ]:
# 3. Hàm chạy 1 lô, kết quả gom vào RES. Bỏ qua ca đã xong (cả trong output version cũ).
def done_ids():
    found = set()
    for p in list(RES.glob('step2_output/*.nii*')) + \
             [Path(x) for x in glob.glob('/kaggle/input/**/step2_output/*.nii*', recursive=True)]:
        found.add(p.name[:4])          # so sánh theo mã BN, không theo đuôi file
    return found

def copy_previous():
    # kết quả từ version trước (nếu đã Add làm Input) -> chép vào RES để zip cuối gom đủ
    for k in KEEP:
        for f in glob.glob(f'/kaggle/input/**/tss_sag/{k}/*', recursive=True):
            dst = RES / k / Path(f).name
            if not dst.exists():
                dst.parent.mkdir(parents=True, exist_ok=True)
                shutil.copy2(f, dst)

def run_batch(files, log):
    if TMP.exists():
        shutil.rmtree(TMP)
    (TMP / 'in').mkdir(parents=True)
    for f in files:
        os.symlink(f, TMP / 'in' / f.name)
    t0 = time.time()
    cmd = ['totalspineseg', str(TMP / 'in'), str(TMP / 'out'), '--data-dir', str(TSS_DATA),
           '--device', 'cuda', '--quiet', '--keep-only', *KEEP]
    r = subprocess.run(cmd, capture_output=True, text=True)
    secs = time.time() - t0
    ok = r.returncode == 0
    if ok:
        for k in KEEP:
            (RES / k).mkdir(parents=True, exist_ok=True)
            for f in (TMP / 'out' / k).glob('*'):
                shutil.move(str(f), RES / k / f.name)
    else:
        print(r.stdout[-2000:], r.stderr[-3000:])
    with open(log, 'a') as fh:
        fh.write(f"{files[0].name},{files[-1].name},{len(files)},{r.returncode},{secs:.1f}\n")
    print(f"{files[0].name}..{files[-1].name}: {len(files)} ca, {secs:.0f} s "
          f"({secs/len(files):.1f} s/ca), {'OK' if ok else 'LỖI ' + str(r.returncode)}")
    return ok

RES.mkdir(parents=True, exist_ok=True)
LOG = RES / 'batch_log.csv'
if not LOG.exists():
    LOG.write_text('first,last,n,exit_code,seconds\n')
copy_previous()
print('Đã xong từ trước:', len(done_ids()))


In [ ]:
# 4. Chạy thử 3 ca đầu (lần đầu tải trọng số ~4 GB nên chậm hơn)
todo = [f for f in inputs if f.name[:4] not in done_ids()]
run_batch(todo[:3], LOG)


In [ ]:
# 5. Chạy toàn bộ (theo lô), dừng nhận lô mới khi gần hết 12 giờ
t_start = time.time()
todo = [f for f in inputs if f.name[:4] not in done_ids()]
print('Cần chạy:', len(todo))
for i in range(0, len(todo), BATCH):
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print('Hết ngân sách thời gian - dừng. Chạy version mới để làm tiếp.')
        break
    run_batch(todo[i:i + BATCH], LOG)
print('Tổng đã xong:', len(done_ids()), '/', len(inputs))


In [ ]:
# 6. Nén kết quả để tải về (tab Output)
missing = sorted(set(f.name[:4] for f in inputs) - done_ids())
print('Thiếu:', len(missing), missing[:20])
shutil.make_archive(str(WORK / 'tss_sag_results'), 'zip', root_dir=WORK, base_dir='tss_sag')
print('Đã tạo', WORK / 'tss_sag_results.zip',
      f"{(WORK / 'tss_sag_results.zip').stat().st_size / 2**20:.0f} MB")
